# PC-FNO3D NATIONAL — OS6 : réseau ANEO sur toute la Tunisie, candidat D (Colab, A100)

**À lancer uniquement avec l'accord écrit d'Ines.** Règle `configs/national/regle_os6_national.json`
(sha256 `c705aa8b9b0c…`, statut : PROPOSITION tant qu'Ines ne l'a pas validée), vérifiée par le carnet
avant tout calcul ; code du dépôt au commit figé `a0c04be1eba4`. **Aucun entraînement.**

Expériences de simulation de systèmes d'observation (OSSE) sur la vérité d'entraînement v3 :

1. **modèle** : le candidat D (`runs/national_1km_v4_D_s<graine>`), graine choisie sur la **validation**
   (minimum de `val_score` de `history.json`), écrite dans `choix_modele.json` ; le run_dir de D n'est que lu ;
2. **capteur ANEO simulé** comme à l'entraînement de D : vérité d'entraînement à
   25 m au-dessus du sol + bruit de mesure 0,1 m/s + 2 % de la vitesse, entré comme
   une station (ce ne sont pas des mesures ; aucun bruit ajouté aux prédictions) ;
3. **candidats** sur toute la Tunisie : terre, pente ≤ 20°, altitude ≤ 650 m, à 1 km du bord, des
   aéroports et entre capteurs ; zones exclues seulement si `os6_zones_exclues.geojson` est déposé ;
4. **placement glouton** sur la validation (24 instants, 5 capteurs par étape,
   jusqu'à 400) ;
5. **courbe de dimensionnement** sur le **test 2022, lu une fois** : optimisé, grille régulière, hasard, pour
   10, 20, 30, 50, 75, 100, 150, 200, 300, 400 capteurs ; nombre recommandé = plus petit nombre dont le RMSE u, v
   maximal sur 0–1 km au-dessus du sol est ≤ 0,5 m/s ; courbe ERA5 dégradée en information ;
6. **plan** : `MyDrive/pcfno3d_national/runs/os6_national_D_s<graine>/` (`plan_aneo.csv`, `.geojson`,
   `courbe_test.csv`, `carte.png`, `os6.json`…), au format du Grand Tunis (onglet admin de la phase 8).

Ressources estimées (non mesurées) : A100 40 Go, ~20–24 Go de GPU par lot de 4 dispositions, ~3 h 30
(2 à 3 fois plus sur L4). Reprise : une étape déjà écrite est relue. La session s'arrête à la fin.

## 1. Drive, code au commit figé, règles vérifiées, données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")   # code et poids du Grand Tunis
NAT = Path("/content/drive/MyDrive/pcfno3d_national")
NAT.mkdir(parents=True, exist_ok=True)
JOURNAUX = NAT / "journaux"
JOURNAUX.mkdir(exist_ok=True)
libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"place libre sur le Drive : {libre:.0f} Go (le jeu national en demande ~50)")

CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille nationale, grille_reelle.json, forcage ERA5 (une archive sur le Drive)
DON = Path("/content/donnees")
G = DON / "national"
if not (G / "forcage_era5.csv").exists():
    with zipfile.ZipFile(NAT / "national_donnees.zip") as zf:
        zf.extractall(DON)
print(sorted(p.name for p in G.iterdir()))

# code du depot public au commit fige (remplace l'archive du Drive, plus ancienne)
COMMIT = "a0c04be1eba48e1e96628574cf519c589d3089ba"
os.chdir("/content")    # le noyau etait dans /content/PFE : le supprimer laisserait pip dans un dossier efface
subprocess.run(f"rm -rf /content/PFE && git init -q /content/PFE && cd /content/PFE && git remote add origin "
               f"https://github.com/ines-rouatbi/pcfno3d.git && git fetch -q --depth 1 origin {COMMIT} "
               f"&& git checkout -q FETCH_HEAD", shell=True, check=True)
os.chdir("/content/PFE")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "/content/PFE"],
                   capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install -e /content/PFE a echoue (sortie ci-dessus)")
import importlib
importlib.invalidate_caches()
import hashlib
for f, h in (("regle_v4_national.json", "60671e843ee648ee2422168a7659a53dfe8aa414be7b1c7c6f54eeec622a43f8"),):
    assert hashlib.sha256((CODE / "configs" / "national" / f).read_bytes()).hexdigest() == h, f
print("code", COMMIT, "| regle v4 verifiee")

# regle OS6 nationale (PROPOSITION tant qu'Ines ne l'a pas validee) : empreinte inscrite dans ce carnet, verifiee ici
from pcfno3d import os6
REGLE = CODE / "configs" / "national" / "regle_os6_national.json"
regle, sha_os6 = os6.charger_regle(REGLE)
assert sha_os6 == "c705aa8b9b0c483ee289d3b85642ed9c959a90bf4c60dad047dfb5ac223cac85", f"regle OS6 modifiee depuis la generation du carnet : {sha_os6}"
print("regle OS6 verifiee :", sha_os6[:12], "|", regle["statut"][:120])

## 2. Cache de la vérité v3

In [ ]:
LOC = Path("/content/national")
CACHE, JEU = LOC / "cache", LOC / "jeu"
for k in (1, 2, 3):
    if not (NAT / f"cache_v3_partie_{k}.zip").exists():
        raise FileNotFoundError(f"cache_v3_partie_{k}.zip absent : finir le carnet de generation")
if not (CACHE / "index.csv").exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    for k in (1, 2, 3):
        with zipfile.ZipFile(NAT / f"cache_v3_partie_{k}.zip") as zf:
            zf.extractall(CACHE)
    with zipfile.ZipFile(NAT / "jeu_v3.zip") as zf:
        zf.extractall(JEU)
    import pandas as pd
    idx = pd.concat([pd.read_csv(CACHE / f"index_partie_{k}.csv") for k in (1, 2, 3)])
    idx.drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
import pandas as pd
print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())
lancer("controle", "python -u -m pcfno3d -c national_1km_colab check")

## 3. GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "pas de GPU : choisir A100 (ou L4) dans Execution > Modifier le type d'execution"
nom_gpu = torch.cuda.get_device_name(0)
total = torch.cuda.get_device_properties(0).total_memory / 1e9
# estimation (non mesuree, regle_os6_national.json) : ~5-6 Go par disposition a 16 x 828 x 388 ;
# un lot trop grand est de toute facon recoupe en deux (WindFieldModel._members)
LOT = 4 if total >= 39 else 2
print(f"{nom_gpu}, {total:.0f} Go | {LOT} dispositions par passage du reseau")

## 4. Graine de D (validation), placement, courbe (test, une fois), plan — puis fin de la session

In [ ]:
import shlex
def choisir_poids(src):
    """Le Drive garde parfois deux fichiers du meme nom (grand_tunis_v4_dso : deux best.pt de 1,69 Go,
    vus le 31/10) ; Colab peut n'en montrer qu'un, au hasard. On retient celui dont l'epoque est celle
    du test evalue (eval_pcfno3d/best_wcont, sinon best), meme s'il n'y en a qu'un : jamais un poids
    pris au hasard."""
    cands = sorted(src.glob("best*.pt"))
    if not cands:
        return src / "best.pt"
    attendu = next((json.loads(f.read_text()).get("epoque") for d in ("best_wcont", "best")
                    if (f := src / "eval_pcfno3d" / d / "test_summary.json").exists()), None)
    import torch
    epoques = {}
    for c in cands:
        ck = torch.load(str(c), map_location="cpu", mmap=True, weights_only=False)
        epoques[c] = ck.get("epoch", -2) + 1
        del ck
    print("poids candidats :", {c.name: e for c, e in epoques.items()}, "| epoque evaluee :", attendu)
    ok = [c for c in cands if epoques[c] == attendu] if attendu is not None else cands
    if not ok:
        raise RuntimeError(f"aucun poids de {src} n'a l'epoque evaluee ({attendu}) : verifie sur le Drive")
    return max(ok, key=lambda c: c.stat().st_mtime)          # a epoque egale : le plus recent

import shlex
import traceback
import yaml
from google.colab import runtime
P = regle["parametres"]
try:
    # 1. graine de D : minimum de val_score de la VALIDATION (history.json) ; le test n'est pas lu
    choix = os6.meilleure_graine(NAT / "runs", "national_1km_v4_D", (42, 43, 44))
    choix["regle_os6_sha256"] = sha_os6
    print(json.dumps(choix, indent=1))
    RUN = Path(choix["run_dir"])
    SORTIE = NAT / "runs" / f"os6_national_D_s{choix['graine']}"      # dossier neuf, hors du run_dir de D
    SORTIE.mkdir(parents=True, exist_ok=True)
    F_CHOIX = SORTIE / "choix_modele.json"
    if F_CHOIX.exists() and json.loads(F_CHOIX.read_text())["graine"] != choix["graine"]:
        raise RuntimeError(f"{F_CHOIX} porte une autre graine : ne rien melanger, voir avec Ines")
    F_CHOIX.write_text(json.dumps(choix, indent=2))
    # 2. configuration : celle du national, run_dir = celui de D (lu seulement : poids et norm_stats.json)
    c = yaml.safe_load(open(CODE / "configs" / "national_1km_colab.yaml"))
    c["name"] = "national_os6_D"
    c["paths"]["run_dir"], c["paths"]["local_dir"] = str(RUN), "/content/run_local_os6_D"
    yaml.safe_dump(c, open(CODE / "configs" / "national_os6_D.yaml", "w"), sort_keys=False)
    CKPT = shlex.quote(str(choisir_poids(RUN)))
    # 3. courbe d'information (ERA5 degradee, parametres de la phase 3, jours d'entrainement)
    DEG = NAT / "runs" / "degradation_era5_v4.json"
    if P["courbe_information_era5_degradee"] and not DEG.exists():
        raise FileNotFoundError(f"{DEG} absent : la regle demande la courbe d'information")
    ZONES = NAT / "os6_zones_exclues.geojson"          # fourni par Tukhnanutha, facultatif ; aucune zone inventee
    print("zones exclues :", ZONES if ZONES.exists() else "aucune")
    # 4. placement (validation), courbe (test, une fois ; une reprise relit les fichiers), plan
    lancer("os6_national",
           f"python -u scripts/os6_reseau_aneo.py -c national_os6_D --checkpoint {CKPT} "
           f"--regle {shlex.quote(str(REGLE))} --modele-info {shlex.quote(str(F_CHOIX))} "
           f"--sortie {shlex.quote(str(SORTIE))} --lot {LOT}"
           + (f" --degradation {shlex.quote(str(DEG))}" if P["courbe_information_era5_degradee"] else "")
           + (f" --zones-exclues {shlex.quote(str(ZONES))}" if ZONES.exists() else ""))
    r = json.loads((SORTIE / "os6.json").read_text())
    print(json.dumps({k: r.get(k) for k in ("nombre_recommande", "metrique_seuil", "candidats",
                                             "capteurs_places", "reseau_actuel", "comparaison_net1_sol")},
                     indent=1, ensure_ascii=False))
    import pandas as pd
    print(pd.read_csv(SORTIE / "courbe_test.csv").round(3).to_string(index=False))
    if (SORTIE / "courbe_test_era5_degradee.csv").exists():
        print("information (ERA5 degradee, sans effet sur le nombre recommande) :")
        print(pd.read_csv(SORTIE / "courbe_test_era5_degradee.csv").round(3).to_string(index=False))
except Exception:
    traceback.print_exc()
finally:
    runtime.unassign()